# Lecture 3.1 — File Tools: Read, Write, Edit

**Course:** Claude Agent SDK: Build AI Agents in Python  
**Section:** 3 — Built-In Tools Deep Dive

---

In this notebook we explore the three built-in file tools — the tools that almost every real-world agent depends on.

| Tool | What it does |
|------|--------------|
| **Read** | Opens any file and brings its full contents into the agent's context |
| **Write** | Creates a new file based on your prompt |
| **Edit** | Makes precise, targeted changes to a specific part of an existing file |

Each tool gets its own isolated demo. We lock `allowed_tools` to a single tool per agent so you can see exactly what each one does on its own.

> **This notebook is self-contained.** No previous lecture notebook is required.

In [1]:
# Install the Claude Agent SDK

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 MB 10.6 MB/s eta 0:00:00


In [2]:
# Load API key from Colab Secrets
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Model Configuration

The cell below sets the model used by every agent in this notebook.

You can change `MODEL_NAME` to any Claude model you have access to.  
For the full list of available models, visit:  
https://platform.claude.com/docs/en/about-claude/models/overview

In [3]:
# Model configuration
# Change this to use a different Claude model
# For the latest available models visit:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

---

## Creating Sample Files for This Lecture

Before we run any agents, we need files to work with. We create two files programmatically:

- **`notes.txt`** — used for the **Read** demo
- **`config.txt`** — used for the **Edit** demo

We do **not** pre-create a file for the Write demo — the agent creates that file itself. That is the point of Write.

In [4]:
import os

os.makedirs("/content/file_tools_demo", exist_ok=True)

# File for Read demo
with open("/content/file_tools_demo/notes.txt", "w") as f:
    f.write("Meeting Notes - March 2025\n")
    f.write("Attendees: Alice, Bob, Carol\n\n")
    f.write("Action Items:\n")
    f.write("- Alice to finish the authentication module by Friday\n")
    f.write("- Bob to write unit tests for the payments module\n")
    f.write("- Carol to update the README with setup instructions\n\n")
    f.write("Next meeting: March 20, 2025\n")

# File for Edit demo
with open("/content/file_tools_demo/config.txt", "w") as f:
    f.write("app_name=FileToolsDemo\n")
    f.write("version=1.0.0\n")
    f.write("debug=false\n")
    f.write("max_connections=10\n")
    f.write("log_level=INFO\n")

# No file pre-created for Write — the agent creates it

## Verifying What Was Created

Good practice: let's see exactly what is on disk before the agents touch anything.

In [5]:
import os

# Show directory structure
print("Directory structure:")
print("=" * 40)
for root, dirs, files in os.walk("/content/file_tools_demo"):
    level = root.replace("/content/file_tools_demo", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

# Show file contents
files_to_show = [
    "/content/file_tools_demo/notes.txt",
    "/content/file_tools_demo/config.txt",
]

print("\nFile contents:")
print("=" * 40)
for filepath in files_to_show:
    print(f"\n--- {filepath} ---")
    with open(filepath, "r") as f:
        print(f.read())

Directory structure:
file_tools_demo/
  config.txt
  notes.txt

File contents:

--- /content/file_tools_demo/notes.txt ---
Meeting Notes - March 2025
Attendees: Alice, Bob, Carol

Action Items:
- Alice to finish the authentication module by Friday
- Bob to write unit tests for the payments module
- Carol to update the README with setup instructions

Next meeting: March 20, 2025


--- /content/file_tools_demo/config.txt ---
app_name=FileToolsDemo
version=1.0.0
debug=false
max_connections=10
log_level=INFO



---

## Part 1 — The Read Tool

**What it does:** Opens any file in the working directory and brings its full contents into the agent's context window. Claude can then reason about that content — summarise it, extract structured information, answer questions about it. You write zero file-handling code: no `open()`, no `read()`, no parsing.

**When to use it:** Any time you want the agent to work with the contents of an existing file.

> Students who went through Section 2 have already used Read — the File Explorer Agent used it alongside Glob. This demo goes deeper: we give the agent a structured analytical task with three specific things to extract.

In [6]:
from claude_agent_sdk import query, ClaudeAgentOptions

async for message in query(
    prompt="""
    Read the file at /content/file_tools_demo/notes.txt
    and give me a concise summary of:
    1. Who attended the meeting
    2. What the action items are and who owns each one
    3. When the next meeting is
    Present as a clean structured summary.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Read"],
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

# Expected output (approximate):
# Meeting Summary
# ===============
# Attendees: Alice, Bob, Carol
#
# Action Items:
# - Alice: Complete authentication module by Friday
# - Bob: Write unit tests for payments module
# - Carol: Update README with setup instructions
#
# Next Meeting: March 20, 2025

## Summary of Meeting Notes

**Attendees:**
- Alice
- Bob
- Carol

**Action Items:**
| Owner | Task |
|-------|------|
| Alice | Finish the authentication module by Friday |
| Bob | Write unit tests for the payments module |
| Carol | Update the README with setup instructions |

**Next Meeting:**
- March 20, 2025


---

## Part 2 — The Write Tool

**What it does:** Creates a new file. You describe the content you want, and the agent writes the file.

**⚠️ Critical warning:** If you use Write on a file that **already exists**, it will **overwrite the entire file**. Not append. Not merge. Overwrite — everything that was there is gone, replaced by whatever the agent generates.

**When to use it:** Creating a brand new file that does not yet exist.

In [7]:
async for message in query(
    prompt="""
    Create a new file at /content/file_tools_demo/summary.txt
    with the following content:
    - A title: "Project Summary"
    - A brief description: "This project contains a file tools demonstration
      with notes and configuration files."
    - A list of files in the project: notes.txt and config.txt
    - Today's date
    Format it as a clean readable document.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Write"],
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

Done! I've created the summary.txt file at `/content/file_tools_demo/summary.txt` with a clean, readable format that includes:

✓ **Title**: "Project Summary"
✓ **Description**: Details about the file tools demonstration
✓ **File listing**: Both notes.txt and config.txt
✓ **Date**: Today's date (June 6, 2026)

The document is formatted with clear sections and visual separators for easy readability.


## Verifying the File Was Created

The agent says it created the file. Let's verify that independently — plain Python, no agent.

In [8]:
with open("/content/file_tools_demo/summary.txt", "r") as f:
    print(f.read())

# Expected output (approximate):
# Project Summary
# ===============
# This project contains a file tools demonstration
# with notes and configuration files.
#
# Files:
# - notes.txt
# - config.txt
#
# Date: March 2025

                            PROJECT SUMMARY

PROJECT DESCRIPTION
-------------------
This project contains a file tools demonstration with notes and configuration
files.


PROJECT FILES
-------------
• notes.txt
• config.txt


DOCUMENT INFORMATION
--------------------
Created: June 6, 2026




---

## Part 3 — The Edit Tool

**What it does:** Makes precise, surgical changes to a specific part of an existing file. Only the targeted part changes — everything else is left exactly as it was.

**How it differs from Write:**
- Write touches the entire file
- Edit touches only what you tell it to

**When to use it:** Any time you want to change one or two lines, update a specific value, or add a section — without disturbing the rest of the file.

We are going back to `config.txt`. It currently has five key-value pairs. We will ask the agent to change exactly two of them — `debug` and `max_connections` — while leaving the other three completely untouched.

In [9]:
async for message in query(
    prompt="""
    In the file /content/file_tools_demo/config.txt,
    make the following changes:
    1. Change the value of debug from false to true
    2. Change the value of max_connections from 10 to 50
    Do not change anything else in the file.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Edit"],
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

Perfect! I've successfully made both changes to `/content/file_tools_demo/config.txt`:
- ✅ Changed `debug=false` to `debug=true`
- ✅ Changed `max_connections=10` to `max_connections=50`

The rest of the file remains unchanged. The configuration now has debug enabled and supports up to 50 connections.


## Verifying Only the Targeted Lines Changed

This is the most instructive moment in the lecture. We read `config.txt` back with plain Python.  
**Two lines should have changed. Three should be completely untouched.**

In [10]:
with open("/content/file_tools_demo/config.txt", "r") as f:
    print(f.read())

# Expected output:
# app_name=FileToolsDemo   <-- unchanged
# version=1.0.0            <-- unchanged
# debug=true               <-- CHANGED (was false)
# max_connections=50       <-- CHANGED (was 10)
# log_level=INFO           <-- unchanged

app_name=FileToolsDemo
version=1.0.0
debug=true
max_connections=50
log_level=INFO



---

## Write vs Edit — The Critical Distinction

Every time your agent needs to work with a file, ask yourself one question:

> **Am I creating something new, or am I changing something that already exists?**

| Situation | Use |
|-----------|-----|
| Creating a brand new file | **Write** |
| Replacing an entire file's content | **Write** |
| Changing one or two lines in an existing file | **Edit** |
| Adding a section to an existing file | **Edit** |
| You want everything else preserved | **Edit** |

### The danger of using Write when you mean Edit

If `config.txt` had 200 lines instead of five, and you used Write to change one value — those 199 other lines would be gone. The agent generates what the prompt asks for; it does not preserve what it does not know about.

Edit is safe precisely because it **only touches what you target**.

**The rule:** New file → `Write`. Existing file, targeted change → `Edit`.

---

## Summary

In this lecture we covered the three core file tools:

- **Read** — Opens any file and brings its full contents into the agent's context. Zero file-handling code on your side.
- **Write** — Creates new files from your prompt. Overwrites the entire file if it already exists — use it for creation only.
- **Edit** — Makes surgical changes to a specific part of an existing file. Only the targeted lines change; everything else is preserved.

**The most important takeaway:** New file → Write. Existing file, targeted change → Edit.

---

**Up next: Lecture 3.2 — Shell Tools: Bash & Monitor**  
We move from files to commands — running terminal operations, executing scripts, triggering git operations, and reacting to live output line by line.